In [1]:
import polars as pl
import duckdb
import plotly.express as px
import plotly.graph_objects as go

from pathlib import Path

print("Library berhasil diimport.")

Library berhasil diimport.


In [2]:
DATA_PATH = "../data/processed/indonesia_palm_oil_cleaned.parquet"

print("Dataset:", DATA_PATH)
print("File tersedia:", Path(DATA_PATH).exists())

Dataset: ../data/processed/indonesia_palm_oil_cleaned.parquet
File tersedia: True


In [3]:
df = pl.scan_parquet(DATA_PATH)

print("Dataset berhasil dibaca dengan Polars Lazy API.")
print("Jumlah kolom:", len(df.collect_schema().names()))

Dataset berhasil dibaca dengan Polars Lazy API.
Jumlah kolom: 34


In [4]:
dataset_info = df.select([
    pl.len().alias("jumlah_baris")
]).collect()

dataset_info

jumlah_baris
u32
1694907


In [5]:
volume_by_year = (
    df
    .group_by("year")
    .agg(
        pl.col("volume").sum().alias("total_volume")
    )
    .sort("year")
    .collect()
)

volume_by_year

year,total_volume
i64,f64
2013,2.7144e7
2014,2.9265e7
2015,3.1070e7
2016,3.1731e7
2017,3.7965e7
2018,4.2878e7
2019,4.7114e7
2020,4.4757e7
2021,4.5025e7


In [6]:
fig = px.line(
    volume_by_year,
    x="year",
    y="total_volume",
    markers=True,
    title="Tren Total Volume Produksi Kelapa Sawit Indonesia 2013–2022",
    labels={
        "year": "Tahun",
        "total_volume": "Total Volume Produksi"
    }
)

fig.update_layout(
    template="plotly_white",
    xaxis=dict(dtick=1),
    hovermode="x unified"
)

fig.show()

In [7]:
volume_by_province = (
    df
    .filter(
        pl.col("province_of_production") != "UNKNOWN"
    )
    .group_by("province_of_production")
    .agg(
        pl.col("volume").sum().alias("total_volume")
    )
    .sort("total_volume", descending=True)
    .collect()
)

volume_by_province.head(10)

province_of_production,total_volume
str,f64
"""RIAU""",3.9191e7
"""KALIMANTAN TENGAH""",3.5659e7
"""SUMATERA UTARA""",3.3933e7
"""KALIMANTAN BARAT""",2.4471e7
"""KALIMANTAN TIMUR""",1.9918e7
"""SUMATERA SELATAN""",1.6804e7
"""JAMBI""",1.3229e7
"""KALIMANTAN SELATAN""",8.7949e6
"""SUMATERA BARAT""",7.4500e6


In [8]:
fig = px.bar(
    volume_by_province.head(10),
    x="total_volume",
    y="province_of_production",
    orientation="h",
    title="10 Provinsi dengan Total Volume Produksi Terbesar",
    labels={
        "total_volume": "Total Volume Produksi",
        "province_of_production": "Provinsi"
    }
)

fig.update_layout(
    template="plotly_white",
    yaxis={"categoryorder": "total ascending"}
)

fig.show()

In [9]:
volume_by_product = (
    df
    .group_by("product_type")
    .agg(
        pl.col("volume").sum().alias("total_volume")
    )
    .sort("total_volume", descending=True)
    .collect()
)

volume_by_product

product_type,total_volume
str,f64
"""PALM OIL""",2.7388e8
"""REFINED PALM OIL""",1.0976e8


In [10]:
fig = px.bar(
    volume_by_product,
    x="product_type",
    y="total_volume",
    text="total_volume",
    title="Perbandingan Total Volume Berdasarkan Product Type",
    labels={
        "product_type": "Jenis Produk",
        "total_volume": "Total Volume Produksi"
    }
)

fig.update_traces(
    texttemplate="%{y:.2s}",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white"
)

fig.show()

In [11]:
volume_fob_province = (
    df
    .filter(
        (pl.col("province_of_production") != "UNKNOWN") &
        pl.col("fob").is_not_null()
    )
    .group_by("province_of_production")
    .agg([
        pl.col("volume").sum().alias("total_volume"),
        pl.col("fob").sum().alias("total_fob")
    ])
    .sort("total_volume", descending=True)
    .collect()
)

volume_fob_province.head(10)

province_of_production,total_volume,total_fob
str,f64,f64
"""RIAU""",3.6010e7,2.7546e10
"""SUMATERA UTARA""",2.7602e7,1.9083e10
"""KALIMANTAN TENGAH""",2.5945e7,1.4455e10
"""KALIMANTAN BARAT""",1.7291e7,9.6250e9
"""KALIMANTAN TIMUR""",1.4756e7,1.0662e10
"""SUMATERA SELATAN""",1.2034e7,8.4833e9
"""JAMBI""",1.1282e7,8.8572e9
"""SUMATERA BARAT""",6.5589e6,4.4771e9
"""KALIMANTAN SELATAN""",5.6691e6,3.4514e9


In [13]:
fig = px.scatter(
    volume_fob_province,
    x="total_volume",
    y="total_fob",
    hover_name="province_of_production",
    title="Hubungan Total Volume Produksi dan FOB per Provinsi",
    labels={
        "total_volume": "Total Volume Produksi",
        "total_fob": "Total FOB"
    }
)

fig.update_layout(
    template="plotly_white"
)

fig.show()

In [14]:
deforestation_by_province = (
    df
    .filter(
        (pl.col("province_of_production") != "UNKNOWN") &
        pl.col("palm_oil_deforestation_10_year_total_exposure").is_not_null()
    )
    .group_by("province_of_production")
    .agg(
        pl.col("palm_oil_deforestation_10_year_total_exposure")
        .sum()
        .alias("total_deforestation_exposure")
    )
    .sort("total_deforestation_exposure", descending=True)
    .collect()
)

deforestation_by_province.head(10)

province_of_production,total_deforestation_exposure
str,f64
"""KALIMANTAN BARAT""",1.9493e6
"""KALIMANTAN TENGAH""",1.2485e6
"""KALIMANTAN TIMUR""",922590.206247
"""KALIMANTAN UTARA""",594139.366252
"""RIAU""",553370.339946
"""PAPUA""",426890.152802
"""PAPUA TENGAH""",272444.872236
"""SUMATERA UTARA""",265413.646593
"""KALIMANTAN SELATAN""",145145.033244


In [15]:
fig = px.bar(
    deforestation_by_province.head(10),
    x="total_deforestation_exposure",
    y="province_of_production",
    orientation="h",
    title="10 Provinsi dengan Total Deforestation Exposure Terbesar",
    labels={
        "total_deforestation_exposure": "Total Deforestation Exposure",
        "province_of_production": "Provinsi"
    }
)

fig.update_layout(
    template="plotly_white",
    yaxis={"categoryorder": "total ascending"}
)

fig.show()

In [16]:
emission_by_year = (
    df
    .filter(
        pl.col("total_emission_exposure").is_not_null()
    )
    .group_by("year")
    .agg([
        pl.col("total_emission_exposure").sum().alias("total_emission"),
        pl.col("net_emission_luc_exposure").sum().alias("total_net_emission"),
        pl.len().alias("jumlah_data")
    ])
    .sort("year")
    .collect()
)

emission_by_year

year,total_emission,total_net_emission,jumlah_data
i64,f64,f64,u32
2021,1.6120e8,8.8335e6,385389
2022,1.6197e8,9.1491e6,284228


In [17]:
fig = px.bar(
    emission_by_year,
    x="year",
    y="total_emission",
    text="total_emission",
    title="Total Emission Exposure Tahun 2021–2022",
    labels={
        "year": "Tahun",
        "total_emission": "Total Emission Exposure"
    }
)

fig.update_traces(
    texttemplate="%{y:.3s}",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    xaxis=dict(dtick=1)
)

fig.show()

In [18]:
# Tahun dengan volume produksi terbesar
max_year = volume_by_year.sort(
    "total_volume",
    descending=True
).head(1)

# Provinsi dengan volume terbesar
max_province = volume_by_province.head(1)

# Product type dengan volume terbesar
max_product = volume_by_product.head(1)

# Provinsi dengan deforestation exposure terbesar
max_deforestation = deforestation_by_province.head(1)

# Tahun dengan emission exposure terbesar
max_emission = emission_by_year.sort(
    "total_emission",
    descending=True
).head(1)

print("=== TAHUN DENGAN VOLUME TERBESAR ===")
print(max_year)

print("\n=== PROVINSI DENGAN VOLUME TERBESAR ===")
print(max_province)

print("\n=== PRODUCT TYPE DENGAN VOLUME TERBESAR ===")
print(max_product)

print("\n=== PROVINSI DENGAN DEFORESTATION EXPOSURE TERBESAR ===")
print(max_deforestation)

print("\n=== TAHUN DENGAN EMISSION TERBESAR ===")
print(max_emission)

=== TAHUN DENGAN VOLUME TERBESAR ===
shape: (1, 2)
┌──────┬──────────────┐
│ year ┆ total_volume │
│ ---  ┆ ---          │
│ i64  ┆ f64          │
╞══════╪══════════════╡
│ 2019 ┆ 4.7114e7     │
└──────┴──────────────┘

=== PROVINSI DENGAN VOLUME TERBESAR ===
shape: (1, 2)
┌────────────────────────┬──────────────┐
│ province_of_production ┆ total_volume │
│ ---                    ┆ ---          │
│ str                    ┆ f64          │
╞════════════════════════╪══════════════╡
│ RIAU                   ┆ 3.9191e7     │
└────────────────────────┴──────────────┘

=== PRODUCT TYPE DENGAN VOLUME TERBESAR ===
shape: (1, 2)
┌──────────────┬──────────────┐
│ product_type ┆ total_volume │
│ ---          ┆ ---          │
│ str          ┆ f64          │
╞══════════════╪══════════════╡
│ PALM OIL     ┆ 2.7388e8     │
└──────────────┴──────────────┘

=== PROVINSI DENGAN DEFORESTATION EXPOSURE TERBESAR ===
shape: (1, 2)
┌────────────────────────┬──────────────────────────────┐
│ province_of_produc

In [19]:
print("===== INSIGHT DATA =====")

print(
    f"1. Volume produksi tahunan tertinggi terjadi pada "
    f"{max_year['year'][0]} dengan total "
    f"{max_year['total_volume'][0]:,.2f}"
)

print(
    f"2. Provinsi dengan total volume produksi terbesar adalah "
    f"{max_province['province_of_production'][0]} dengan total "
    f"{max_province['total_volume'][0]:,.2f}"
)

print(
    f"3. Product type dengan total volume terbesar adalah "
    f"{max_product['product_type'][0]} dengan total "
    f"{max_product['total_volume'][0]:,.2f}"
)

print(
    f"4. Provinsi dengan total deforestation exposure terbesar adalah "
    f"{max_deforestation['province_of_production'][0]} dengan total "
    f"{max_deforestation['total_deforestation_exposure'][0]:,.2f}"
)

print(
    f"5. Tahun dengan total emission exposure terbesar adalah "
    f"{max_emission['year'][0]} dengan total "
    f"{max_emission['total_emission'][0]:,.2f}"
)

===== INSIGHT DATA =====
1. Volume produksi tahunan tertinggi terjadi pada 2019 dengan total 47,113,546.71
2. Provinsi dengan total volume produksi terbesar adalah RIAU dengan total 39,191,423.00
3. Product type dengan total volume terbesar adalah PALM OIL dengan total 273,879,112.78
4. Provinsi dengan total deforestation exposure terbesar adalah KALIMANTAN BARAT dengan total 1,949,347.35
5. Tahun dengan total emission exposure terbesar adalah 2022 dengan total 161,972,673.77


## 5. Insights Utama

Berdasarkan hasil Exploratory Data Analysis (EDA), diperoleh beberapa temuan utama:

### 1. Tren Volume Produksi Berdasarkan Tahun

Volume produksi tahunan tertinggi terjadi pada **2019**, dengan total volume sebesar **47.113.546,71**. Setelah mencapai nilai tersebut, volume produksi mengalami penurunan pada 2020 dan kembali meningkat pada 2021–2022.

### 2. Provinsi dengan Volume Produksi Terbesar

**RIAU** memiliki total volume produksi terbesar dibandingkan provinsi lainnya, yaitu sebesar **39.191.423,00**. Hal ini menunjukkan bahwa Riau merupakan salah satu wilayah dengan kontribusi volume produksi yang besar dalam dataset.

### 3. Product Type dengan Volume Terbesar

Berdasarkan jenis produk, **PALM OIL** memiliki total volume sebesar **273.879.112,78**, lebih besar dibandingkan **REFINED PALM OIL**. Dengan demikian, PALM OIL menjadi *product type* dengan total volume terbesar pada dataset.

### 4. Deforestation Exposure Berdasarkan Provinsi

**KALIMANTAN BARAT** memiliki total *deforestation exposure* terbesar, yaitu **1.949.347,35**, diikuti oleh Kalimantan Tengah dan Kalimantan Timur. Temuan ini menunjukkan adanya perbedaan paparan deforestasi antarwilayah dalam dataset.

### 5. Emission Exposure pada Tahun 2021–2022

Data *total emission exposure* yang tersedia dalam dataset terdapat pada tahun **2021 dan 2022**. Nilai total emission exposure pada **2022** mencapai **161.972.673,77**, lebih tinggi dibandingkan 2021 yang sebesar sekitar **161,20 juta**. Oleh karena itu, perbandingan emisi dalam analisis ini terbatas pada periode 2021–2022.